# 江の島・八景島 精密3Dモデル（`.glb`）生成ノートブック（Google Colab用）

このノートブックは、スナイプ級ヨットレースゲーム（`areyousail-rg/sailing`）向けに、**実測オープンデータ**から精密な **江の島（`enoshima.glb`）** と **八景島（`hakkeijima.glb`）** の軽量3Dモデルを生成・プレビューするためのツールです。

### 使用するオープンデータ（商用・Google Play 公開対応）
1. **国土地理院 標高タイル（DEM5A 5mメッシュ優先 → DEM5B / DEM10B 自動補完）**
   - 実際の島の起伏・海食崖・山二つ・護岸・海岸線を1mスケールで忠実に再現
2. **国土地理院 シームレス空中写真（Zoom 16）**
   - リアルな植生・岩場・砂浜・道路・ヨットハーバーをテクスチャとして焼き込み
3. **実測3D建物データ（2方式に対応）**
   - **自動モード（デフォルト）**: OpenStreetMap（Overpass API）から実測の建物フットプリント・階数/高さ・防波堤・桟橋を自動取得して3D化
   - **PLATEAU CityGML モード（任意）**: 国土交通省 Project PLATEAU（藤沢市・横浜市）の公式 LOD1/LOD2 `.gml` / `.zip` を指定すると、さらに高精度な実測3D建物（屋根形状付き）へ差し替え可能

> **出典表記について（公開時にクレジットへ明記）**  
> ・国土地理院 地理院タイル（標高タイル・シームレス空中写真）を加工して作成  
> ・© OpenStreetMap contributors（OSM建物利用時） / 国土交通省 Project PLATEAU（PLATEAU利用時・CC BY 4.0）

### 座標系（ゲーム本編 `js/world.js` と完全一致）
- `+X = 東 [m]` / `+Y = 上（標高）[m]` / `-Z = 北 [m]`
- レース海面の原点 `(0, 0)` を基準に、江の島は `(+450, -1150)` 付近、八景島は `(-1400, +1180)` 付近に実寸スケールで配置されます。

### 使い方
メニューの **「ランタイム」→「すべてのセルを実行」** を押すだけで、ダウンロード → 3D生成 → Colab内インタラクティブThree.jsプレビュー → `.glb` 書き出しまで完了します（CPUランタイムのままでOK、所要約1〜3分）。

In [ ]:
# @title 1. 必要ライブラリのインストール（初回のみ約20秒）
!pip -q install trimesh mapbox-earcut shapely pillow requests numpy scipy matplotlib lxml
import os, io, math, time, json, base64, zipfile, pathlib
from concurrent.futures import ThreadPoolExecutor, as_completed
import numpy as np
import requests
from PIL import Image, ImageEnhance
import trimesh
from trimesh.visual.material import PBRMaterial
from trimesh.visual import TextureVisuals
from shapely.geometry import Polygon, MultiPolygon
from shapely.ops import unary_union
import matplotlib.pyplot as plt
from IPython.display import display, HTML

os.makedirs("/content/tile_cache", exist_ok=True)
os.makedirs("/content/output", exist_ok=True)
print("✅ ライブラリの準備完了")

In [ ]:
# @title 2. 生成設定（解像度・対象エリア・PLATEAU連携の切替）
# 生成する海面（両方なら ['enoshima', 'hakkeijima']）
TARGET_VENUES = ["enoshima", "hakkeijima"]  # @param ["['enoshima', 'hakkeijima']", "['enoshima']", "['hakkeijima']"] {type:"raw"}

# 地形メッシュのグリッド分割数（大きいほど精密だがファイルサイズ増。推奨: 180〜240）
TERRAIN_GRID_N = 200  # @param {type:"slider", min:100, max:320, step:20}

# 航空写真テクスチャ解像度（1024 / 2048。スマホ向け軽量化なら2048 JPEG品質82が最適）
ORTHO_TEX_SIZE = 2048  # @param [1024, 2048, 4096] {type:"raw"}
ORTHO_JPEG_QUALITY = 82  # @param {type:"slider", min:65, max:95, step:1}

# 実測建物の生成（OSM Overpass API を自動取得）
INCLUDE_OSM_BUILDINGS = True  # @param {type:"boolean"}

# （任意）国土交通省 PLATEAU CityGML（.gml または .zip）を使う場合のパス
# 例: Google Drive をマウントして "/content/drive/MyDrive/plateau_fujisawa.zip" を指定
PLATEAU_PATHS = {
    "enoshima": "",    # 藤沢市（江の島）PLATEAU CityGML .gml/.zip パス（空欄ならOSM自動取得）
    "hakkeijima": "",  # 横浜市（八景島）PLATEAU CityGML .gml/.zip パス（空欄ならOSM自動取得）
}

# ゲーム本編 (js/world.js) と一致させた実世界座標設定
# origin_lat/lon = ゲーム内レース海面の原点 (0, 0) に相当する実緯度経度
# center_lat/lon = 切り出す3D地形エリアの中心緯度経度
VENUES = {
    "enoshima": {
        "label": "江の島・湘南港（藤沢市）",
        "origin_lat": 35.28947,   # 島中心(35.2998, 139.4803) がゲーム座標 (+450, -1150) に一致
        "origin_lon": 139.47535,
        "center_lat": 35.3022,    # 江の島＋弁天橋＋片瀬海岸をバランスよく収録
        "center_lon": 139.4810,
        "half_w_m": 1250.0,       # 東西 2.5km
        "half_h_m": 950.0,        # 南北 1.9km
        "dem_zoom": 15,           # DEM5A (5mメッシュ)
        "ortho_zoom": 16,         # 航空写真 高解像度
        "vertical_exag": 1.08,    # 海上遠望で立体感が映える微補正（1.0で完全実寸）
        "preview_cam": [0.0, 28.0, 0.0],          # レース海面(0,0)からの視点
        "preview_target": [450.0, 28.0, -1150.0], # 江の島シーキャンドル付近を注視
    },
    "hakkeijima": {
        "label": "八景島・福浦・海の公園（横浜市金沢区）",
        "origin_lat": 35.34730,   # 島中心(35.3367, 139.6455) がゲーム座標 (-1400, +1180) に一致
        "origin_lon": 139.66092,
        "center_lat": 35.3360,    # 八景島＋福浦護岸＋海の公園＋夏島周辺を収録
        "center_lon": 139.6450,
        "half_w_m": 1550.0,       # 東西 3.1km
        "half_h_m": 1550.0,       # 南北 3.1km
        "dem_zoom": 15,           # DEM5A (5mメッシュ)
        "ortho_zoom": 16,         # 航空写真 高解像度
        "vertical_exag": 1.05,
        "preview_cam": [0.0, 32.0, 0.0],           # レース海面(0,0)からの視点
        "preview_target": [-1400.0, 22.0, 1180.0], # 八景島シーパラダイス付近を注視
    },
}

print("対象海面:", TARGET_VENUES)

In [ ]:
# @title 3. 国土地理院タイル取得（DEM5A/5B/10B 標高 ＋ シームレス航空写真）
R_EARTH = 6378137.0
SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "AreYouSail-ColabVenueBuilder/1.0 (+https://github.com/areyousail-rg/sailing)"})


def latlon_to_game_xz(lat, lon, origin_lat, origin_lon):
    """緯度経度 -> ゲーム座標 (x=東[m], z=-北[m])"""
    x = np.radians(lon - origin_lon) * R_EARTH * math.cos(math.radians(origin_lat))
    z = -np.radians(lat - origin_lat) * R_EARTH
    return x, z


def game_xz_to_latlon(x, z, origin_lat, origin_lon):
    """ゲーム座標 (x, z) -> 緯度経度"""
    lat = origin_lat + np.degrees(-z / R_EARTH)
    lon = origin_lon + np.degrees(x / (R_EARTH * math.cos(math.radians(origin_lat))))
    return lat, lon


def latlon_to_tile_xy(lat, lon, z):
    """Web Mercator 浮動小数タイル座標 (tx, ty)"""
    n = 2.0 ** z
    tx = (lon + 180.0) / 360.0 * n
    lat_rad = np.radians(lat)
    ty = (1.0 - np.log(np.tan(lat_rad) + 1.0 / np.cos(lat_rad)) / math.pi) / 2.0 * n
    return tx, ty


def fetch_tile_bytes(layer, z, x, y, ext="png"):
    cache_path = f"/content/tile_cache/{layer}_{z}_{x}_{y}.{ext}"
    if os.path.exists(cache_path):
        if os.path.getsize(cache_path) == 0:
            return None
        with open(cache_path, "rb") as f:
            return f.read()
    url = f"https://cyberjapandata.gsi.go.jp/xyz/{layer}/{z}/{x}/{y}.{ext}"
    for attempt in range(3):
        try:
            r = SESSION.get(url, timeout=15)
            if r.status_code == 200 and len(r.content) > 64:
                with open(cache_path, "wb") as f:
                    f.write(r.content)
                return r.content
            if r.status_code == 404:
                # 欠損タイルを空ファイルでキャッシュして再リクエストを防止
                open(cache_path, "wb").close()
                return None
        except Exception:
            time.sleep(0.4 * (attempt + 1))
    return None


def decode_gsi_dem_png(raw_bytes):
    """国土地理院 標高PNGタイルを標高配列 [m] (float32, 欠損=NaN) にデコード"""
    arr = np.asarray(Image.open(io.BytesIO(raw_bytes)).convert("RGB"), dtype=np.int32)
    x = (arr[:, :, 0] << 16) + (arr[:, :, 1] << 8) + arr[:, :, 2]
    h = np.full(x.shape, np.nan, dtype=np.float32)
    valid = x != 8388608  # 2^23 = 欠損(海面等)
    pos = valid & (x < 8388608)
    neg = valid & (x > 8388608)
    h[pos] = x[pos] * 0.01
    h[neg] = (x[neg] - 16777216) * 0.01
    return h


def fetch_dem_tile_with_fallback(z, tx, ty):
    """DEM5A (5m LiDAR) -> DEM5B (5m写真測量) -> DEM10B (10m, z=14を2倍補間) の順で補完"""
    h = np.full((256, 256), np.nan, dtype=np.float32)
    for layer in ("dem5a_png", "dem5b_png"):
        raw = fetch_tile_bytes(layer, z, tx, ty, "png")
        if raw is not None:
            d = decode_gsi_dem_png(raw)
            mask = np.isnan(h) & ~np.isnan(d)
            h[mask] = d[mask]
            if not np.isnan(h).any():
                return h
    # 残り欠損を DEM10B (最大ズーム14) で補完
    if z >= 15:
        dz = z - 14
        scale = 1 << dz
        px, py = tx >> dz, ty >> dz
        ox, oy = tx - (px << dz), ty - (py << dz)
        raw10 = fetch_tile_bytes("dem_png", 14, px, py, "png")
        if raw10 is not None:
            d10 = decode_gsi_dem_png(raw10)
            step = 256 // scale
            sub = d10[oy * step:(oy + 1) * step, ox * step:(ox + 1) * step]
            # 欠損(NaN)を保ったまま拡大補間
            sub_valid = ~np.isnan(sub)
            sub_filled = np.where(sub_valid, sub, 0.0)
            up = np.array(Image.fromarray(sub_filled).resize((256, 256), Image.BILINEAR))
            up_m = np.array(Image.fromarray(sub_valid.astype(np.uint8) * 255).resize((256, 256), Image.NEAREST)) > 127
            mask = np.isnan(h) & up_m
            h[mask] = up[mask]
    else:
        raw10 = fetch_tile_bytes("dem_png", z, tx, ty, "png")
        if raw10 is not None:
            d10 = decode_gsi_dem_png(raw10)
            mask = np.isnan(h) & ~np.isnan(d10)
            h[mask] = d10[mask]
    return h


def build_gsi_raster_mosaic(lat_min, lat_max, lon_min, lon_max, z, mode="dem"):
    """指定緯度経度範囲を覆うタイルを並列取得してモザイク化"""
    tx0f, ty0f = latlon_to_tile_xy(lat_max, lon_min, z)  # 北西
    tx1f, ty1f = latlon_to_tile_xy(lat_min, lon_max, z)  # 南東
    tx0, tx1 = int(math.floor(tx0f)), int(math.floor(tx1f))
    ty0, ty1 = int(math.floor(ty0f)), int(math.floor(ty1f))
    nx, ny = tx1 - tx0 + 1, ty1 - ty0 + 1
    print(f"  [{mode}] z={z}: {nx}x{ny} = {nx*ny} タイル取得中...")

    coords = [(tx, ty) for ty in range(ty0, ty1 + 1) for tx in range(tx0, tx1 + 1)]
    if mode == "dem":
        mosaic = np.full((ny * 256, nx * 256), np.nan, dtype=np.float32)
        with ThreadPoolExecutor(max_workers=6) as ex:
            futs = {ex.submit(fetch_dem_tile_with_fallback, z, tx, ty): (tx, ty) for tx, ty in coords}
            for f in as_completed(futs):
                tx, ty = futs[f]
                r0, c0 = (ty - ty0) * 256, (tx - tx0) * 256
                mosaic[r0:r0 + 256, c0:c0 + 256] = f.result()
        return mosaic, (tx0, ty0, z)
    else:
        mosaic = Image.new("RGB", (nx * 256, ny * 256), (16, 46, 74))
        with ThreadPoolExecutor(max_workers=6) as ex:
            futs = {ex.submit(fetch_tile_bytes, "seamlessphoto", z, tx, ty, "jpg"): (tx, ty) for tx, ty in coords}
            for f in as_completed(futs):
                tx, ty = futs[f]
                raw = f.result()
                if raw is not None:
                    img = Image.open(io.BytesIO(raw)).convert("RGB")
                    mosaic.paste(img, ((tx - tx0) * 256, (ty - ty0) * 256))
        return mosaic, (tx0, ty0, z)


def sample_mosaic_bilinear(mosaic, tile_origin, lat_grid, lon_grid):
    """モザイク配列を (lat_grid, lon_grid) 位置で双線形補間サンプリング"""
    tx0, ty0, z = tile_origin
    tx, ty = latlon_to_tile_xy(lat_grid, lon_grid, z)
    px = np.clip((tx - tx0) * 256.0 - 0.5, 0.0, mosaic.shape[1] - 1.001)
    py = np.clip((ty - ty0) * 256.0 - 0.5, 0.0, mosaic.shape[0] - 1.001)
    x0 = np.floor(px).astype(np.int32)
    y0 = np.floor(py).astype(np.int32)
    fx = px - x0
    fy = py - y0
    v00 = mosaic[y0, x0]
    v10 = mosaic[y0, x0 + 1]
    v01 = mosaic[y0 + 1, x0]
    v11 = mosaic[y0 + 1, x0 + 1]
    # 海面欠損(NaN)は0m扱い
    w00 = (1 - fx) * (1 - fy)
    w10 = fx * (1 - fy)
    w01 = (1 - fx) * fy
    w11 = fx * fy
    num = (
        np.where(np.isnan(v00), 0.0, v00) * w00 +
        np.where(np.isnan(v10), 0.0, v10) * w10 +
        np.where(np.isnan(v01), 0.0, v01) * w01 +
        np.where(np.isnan(v11), 0.0, v11) * w11
    )
    den = (
        (~np.isnan(v00)) * w00 +
        (~np.isnan(v10)) * w10 +
        (~np.isnan(v01)) * w01 +
        (~np.isnan(v11)) * w11
    )
    out = np.where(den > 0.15, num / np.maximum(den, 1e-6), np.nan)
    return out.astype(np.float32)

print("✅ 地理院タイル取得・デコード関数を定義しました")

In [ ]:
# @title 4. 地形メッシュ生成（海面Zファイティング除去・航空写真UV焼き込み）
def crop_ortho_to_bbox(ortho_img, tile_origin, lat_min, lat_max, lon_min, lon_max, out_size):
    """航空写真モザイクを対象領域の正確な正方形テクスチャへ切り出し"""
    tx0, ty0, z = tile_origin
    lx, ty = latlon_to_tile_xy(lat_max, lon_min, z)
    rx, by = latlon_to_tile_xy(lat_min, lon_max, z)
    box = (
        (lx - tx0) * 256.0,
        (ty - ty0) * 256.0,
        (rx - tx0) * 256.0,
        (by - ty0) * 256.0,
    )
    cropped = ortho_img.crop(box).resize((out_size, out_size), Image.LANCZOS)
    # 屋外太陽光下で見栄えが良いようコントラスト・彩度をわずかに整える
    cropped = ImageEnhance.Color(cropped).enhance(1.08)
    cropped = ImageEnhance.Contrast(cropped).enhance(1.04)
    return cropped


def build_terrain_mesh(cfg, grid_n, tex_size):
    """DEM5A + 航空写真から海面トリミング済み地形 Trimesh を生成"""
    cx, cz = latlon_to_game_xz(cfg["center_lat"], cfg["center_lon"], cfg["origin_lat"], cfg["origin_lon"])
    x_min, x_max = cx - cfg["half_w_m"], cx + cfg["half_w_m"]
    z_min, z_max = cz - cfg["half_h_m"], cz + cfg["half_h_m"]  # z_min=北端, z_max=南端

    lat_max, lon_min = game_xz_to_latlon(x_min, z_min, cfg["origin_lat"], cfg["origin_lon"])
    lat_min, lon_max = game_xz_to_latlon(x_max, z_max, cfg["origin_lat"], cfg["origin_lon"])

    dem_mosaic, dem_orig = build_gsi_raster_mosaic(lat_min, lat_max, lon_min, lon_max, cfg["dem_zoom"], mode="dem")
    ortho_mosaic, ortho_orig = build_gsi_raster_mosaic(lat_min, lat_max, lon_min, lon_max, cfg["ortho_zoom"], mode="ortho")

    # 規則グリッド (行 j=0..N: 北->南, 列 i=0..N: 西->東)
    xs = np.linspace(x_min, x_max, grid_n + 1, dtype=np.float32)
    zs = np.linspace(z_min, z_max, grid_n + 1, dtype=np.float32)
    XX, ZZ = np.meshgrid(xs, zs)
    LAT, LON = game_xz_to_latlon(XX, ZZ, cfg["origin_lat"], cfg["origin_lon"])
    H_raw = sample_mosaic_bilinear(dem_mosaic, dem_orig, LAT, LON)

    # 海域(NaNまたは<=0.05m)は -4.5m へ沈めてゲームの3D海面シェーダーとのZファイティングを完全防止
    is_land = (~np.isnan(H_raw)) & (H_raw > 0.05)
    H = np.where(is_land, H_raw * cfg["vertical_exag"], -4.5).astype(np.float32)

    # 頂点座標・UV (trimesh規約: u=0西..1東, v=1北(画像上)..0南(画像下))
    verts = np.stack([XX, H, ZZ], axis=-1).reshape(-1, 3)
    us = np.linspace(0.0, 1.0, grid_n + 1, dtype=np.float32)
    vs = np.linspace(1.0, 0.0, grid_n + 1, dtype=np.float32)
    UU, VV = np.meshgrid(us, vs)
    uvs = np.stack([UU, VV], axis=-1).reshape(-1, 2)

    # 三角形インデックス作成（+Y上向き法線のCCW巻き）
    jj, ii = np.meshgrid(np.arange(grid_n), np.arange(grid_n), indexing="ij")
    v00 = (jj * (grid_n + 1) + ii).ravel()
    v10 = v00 + 1
    v01 = v00 + (grid_n + 1)
    v11 = v01 + 1
    faces = np.vstack([
        np.stack([v00, v01, v10], axis=-1),
        np.stack([v10, v01, v11], axis=-1),
    ])

    # 3頂点すべてが海面下(-3.2m未満)の純海域三角形を間引いてポリゴン数を大幅削減
    f_heights = H.ravel()[faces]
    keep = np.any(f_heights > -3.2, axis=1)
    faces = faces[keep]

    # 未使用頂点を圧縮
    used_idx, inv = np.unique(faces.ravel(), return_inverse=True)
    verts_c = verts[used_idx]
    uvs_c = uvs[used_idx]
    faces_c = inv.reshape(-1, 3)

    ortho_tex = crop_ortho_to_bbox(ortho_mosaic, ortho_orig, lat_min, lat_max, lon_min, lon_max, tex_size)

    # JPEG圧縮画像をマテリアルへ埋め込み（GLBサイズを大幅に軽量化）
    buf = io.BytesIO()
    ortho_tex.save(buf, format="JPEG", quality=ORTHO_JPEG_QUALITY, optimize=True)
    buf.seek(0)
    jpeg_img = Image.open(buf)
    jpeg_img.load()

    mat = PBRMaterial(
        name=f"{cfg['label']}_terrain",
        baseColorTexture=jpeg_img,
        roughnessFactor=0.92,
        metallicFactor=0.0,
    )
    mesh = trimesh.Trimesh(
        vertices=verts_c,
        faces=faces_c,
        visual=TextureVisuals(uv=uvs_c, material=mat),
        process=False,
    )
    mesh.fix_normals()

    def sample_h(x_arr, z_arr):
        la, lo = game_xz_to_latlon(np.asarray(x_arr), np.asarray(z_arr), cfg["origin_lat"], cfg["origin_lon"])
        hv = sample_mosaic_bilinear(dem_mosaic, dem_orig, la, lo)
        return np.where(np.isnan(hv), 0.0, hv * cfg["vertical_exag"])

    meta = {
        "bbox_latlon": (lat_min, lat_max, lon_min, lon_max),
        "bbox_game": (x_min, x_max, z_min, z_max),
        "H_grid": H,
        "ortho_tex": ortho_tex,
        "sample_h": sample_h,
    }
    print(f"  [terrain] 頂点数={len(verts_c):,}, 三角形数={len(faces_c):,}, 最高標高={np.nanmax(H_raw):.1f}m")
    return mesh, meta

print("✅ 地形メッシュ生成関数を定義しました")

In [ ]:
# @title 5. 実測建物ビルダー（OSM Overpass API 自動取得 ＋ PLATEAU CityGML LOD1/LOD2 対応）
OVERPASS_ENDPOINTS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
]


def parse_height_m(tags):
    """OSMタグから建物高さ[m]を推定"""
    for key in ("height", "building:height"):
        if key in tags:
            s = str(tags[key]).lower().replace("m", "").strip().split(";")[0]
            try:
                v = float(s)
                if 1.5 <= v <= 260.0:
                    return v
            except Exception:
                pass
    if "building:levels" in tags:
        try:
            lv = float(str(tags["building:levels"]).split(";")[0].strip())
            if 0.5 <= lv <= 60:
                return max(3.2, lv * 3.3 + 1.0)
        except Exception:
            pass
    btype = tags.get("building", "yes")
    if btype in ("apartments", "hotel", "commercial", "hospital", "university"):
        return 14.0
    if btype in ("industrial", "warehouse", "hangar"):
        return 10.5
    if btype in ("temple", "shrine"):
        return 9.0
    if btype in ("shed", "hut", "garage", "kiosk", "toilets", "roof"):
        return 3.5
    return 6.8


def pick_building_category(tags, height_m):
    """景観に馴染む建築マテリアルカテゴリへ分類"""
    btype = tags.get("building", "")
    name = tags.get("name", "")
    if tags.get("man_made") in ("breakwater", "pier", "groyne", "quay"):
        return "seawall"
    if btype in ("temple", "shrine") or "神社" in name or "寺" in name:
        return "shrine"
    if "シーキャンドル" in name or tags.get("man_made") in ("tower", "lighthouse"):
        return "landmark"
    if btype in ("industrial", "warehouse") or height_m >= 18.0:
        return "concrete_high"
    if btype in ("hotel", "commercial", "hospital", "public", "aquarium") or "アクア" in name or "ヨット" in name:
        return "coastal_white"
    return "residential"


BUILDING_PALETTE = {
    "residential":   ([228, 222, 212, 255], 0.88, 0.02),
    "coastal_white": ([242, 245, 248, 255], 0.72, 0.05),
    "concrete_high": ([198, 206, 214, 255], 0.80, 0.06),
    "shrine":        ([150, 74, 58, 255],   0.85, 0.02),
    "landmark":      ([230, 238, 246, 255], 0.45, 0.25),
    "seawall":       ([172, 174, 170, 255], 0.94, 0.0),
}


def fetch_osm_buildings_and_seawalls(lat_min, lat_max, lon_min, lon_max, cache_key):
    cache_file = f"/content/tile_cache/osm_{cache_key}.json"
    if os.path.exists(cache_file):
        with open(cache_file, "r", encoding="utf-8") as f:
            return json.load(f)
    query = f"""
    [out:json][timeout:45];
    (
      way["building"]({lat_min:.5f},{lon_min:.5f},{lat_max:.5f},{lon_max:.5f});
      way["man_made"~"breakwater|pier|groyne"]({lat_min:.5f},{lon_min:.5f},{lat_max:.5f},{lon_max:.5f});
      way["man_made"~"tower|lighthouse"]({lat_min:.5f},{lon_min:.5f},{lat_max:.5f},{lon_max:.5f});
    );
    (._;>;);
    out body;
    """
    for ep in OVERPASS_ENDPOINTS:
        try:
            r = SESSION.post(ep, data={"data": query}, timeout=60)
            if r.status_code == 200:
                data = r.json()
                with open(cache_file, "w", encoding="utf-8") as f:
                    json.dump(data, f)
                return data
        except Exception as e:
            print(f"  Overpass再試行 ({ep}): {e}")
            time.sleep(1.5)
    print("  ⚠️ Overpass取得に失敗しました（地形のみで継続します）")
    return {"elements": []}


def extrude_building_poly_to_mesh(poly_xz, y_base, height_m):
    """2D (x, z) ポリゴンを y_base .. y_base+height_m に押し出してゲーム座標 Trimesh を返す"""
    try:
        if not poly_xz.is_valid:
            poly_xz = poly_xz.buffer(0)
        if poly_xz.is_empty or poly_xz.area < 6.0:
            return None
        if isinstance(poly_xz, MultiPolygon):
            poly_xz = max(poly_xz.geoms, key=lambda g: g.area)
        # 頂点簡略化で軽量化（0.4mトレランス）
        poly_xz = poly_xz.simplify(0.4, preserve_topology=True)
        if poly_xz.is_empty or poly_xz.area < 6.0:
            return None
        # trimesh.extrude_polygon は XY平面を +Z へ押し出す -> (x, y, z) へ軸変換
        m_xy = trimesh.creation.extrude_polygon(poly_xz, height=height_m, engine="earcut")
        v = m_xy.vertices.copy()
        # poly_xz の座標は (x, -z) で渡して +Z(高さ)->+Y にすると右手系CCWが保たれる
        out_v = np.column_stack([v[:, 0], y_base + v[:, 2], -v[:, 1]]).astype(np.float32)
        return trimesh.Trimesh(vertices=out_v, faces=m_xy.faces, process=False)
    except Exception:
        return None


def build_osm_meshes(cfg, meta, cache_key):
    lat_min, lat_max, lon_min, lon_max = meta["bbox_latlon"]
    data = fetch_osm_buildings_and_seawalls(lat_min, lat_max, lon_min, lon_max, cache_key)
    nodes = {}
    ways = []
    for el in data.get("elements", []):
        if el["type"] == "node":
            nodes[el["id"]] = (el["lat"], el["lon"])
        elif el["type"] == "way" and "tags" in el:
            ways.append(el)

    buckets = {k: [] for k in BUILDING_PALETTE}
    sample_h = meta["sample_h"]

    for w in ways:
        nds = w.get("nodes", [])
        if len(nds) < 3:
            continue
        pts = [nodes[nid] for nid in nds if nid in nodes]
        if len(pts) < 3:
            continue
        tags = w.get("tags", {})
        lats = np.array([p[0] for p in pts])
        lons = np.array([p[1] for p in pts])
        xs, zs = latlon_to_game_xz(lats, lons, cfg["origin_lat"], cfg["origin_lon"])

        is_closed = (nds[0] == nds[-1]) and len(pts) >= 4
        is_seawall = tags.get("man_made") in ("breakwater", "pier", "groyne")
        if is_closed:
            # 右手系保存のため (x, -z) でポリゴン作成
            poly = Polygon(np.column_stack([xs, -zs]))
        elif is_seawall:
            from shapely.geometry import LineString
            width = 6.0 if tags.get("man_made") == "breakwater" else 3.5
            poly = LineString(np.column_stack([xs, -zs])).buffer(width * 0.5)
        else:
            continue

        h_m = 2.6 if is_seawall else parse_height_m(tags)
        cat = pick_building_category(tags, h_m)
        cx, cz = float(np.mean(xs)), float(np.mean(zs))
        ground_y = 0.0 if is_seawall else float(sample_h(cx, cz))
        # 斜面で建物が浮かないよう底面を1.8m地中へ埋め込む
        y_base = -2.2 if is_seawall else (ground_y - 1.8)
        m = extrude_building_poly_to_mesh(poly, y_base, h_m + (2.2 if is_seawall else 1.8))
        if m is not None:
            buckets[cat].append(m)

    out_meshes = {}
    total_b = 0
    for cat, mlist in buckets.items():
        if not mlist:
            continue
        total_b += len(mlist)
        merged = trimesh.util.concatenate(mlist)
        rgba, rough, metal = BUILDING_PALETTE[cat]
        mat = PBRMaterial(
            name=f"bldg_{cat}",
            baseColorFactor=rgba,
            roughnessFactor=rough,
            metallicFactor=metal,
        )
        merged.visual = TextureVisuals(material=mat)
        merged.fix_normals()
        out_meshes[cat] = merged
    print(f"  [OSM建物/防波堤] {total_b:,} 棟を生成（{len(out_meshes)} マテリアルへ統合）")
    return out_meshes


def load_plateau_citygml_meshes(gml_or_zip_path, cfg, meta):
    """国土交通省 Project PLATEAU の CityGML (.gml / .zip) から LOD2/LOD1 実測3D建物を読み込み"""
    from lxml import etree

    if not gml_or_zip_path or not os.path.exists(gml_or_zip_path):
        return None
    x_min, x_max, z_min, z_max = meta["bbox_game"]

    def iter_gml_streams(path):
        if path.lower().endswith(".zip"):
            with zipfile.ZipFile(path, "r") as zf:
                for info in zf.infolist():
                    if info.filename.lower().endswith(".gml") and "bldg" in info.filename.lower():
                        yield info.filename, zf.open(info)
        elif os.path.isdir(path):
            for p in pathlib.Path(path).rglob("*.gml"):
                if "bldg" in p.name.lower():
                    yield p.name, open(p, "rb")
        else:
            yield os.path.basename(path), open(path, "rb")

    all_verts = []
    all_faces = []
    v_offset = 0
    bldg_count = 0

    for fname, stream in iter_gml_streams(gml_or_zip_path):
        print(f"  [PLATEAU] 解析中: {fname}")
        ctx = etree.iterparse(stream, events=("end",), tag="{http://www.opengis.net/citygml/building/2.0}Building")
        for _, elem in ctx:
            # LOD2 優先、無ければ LOD1 を採用
            pos_lists = elem.xpath(".//*[local-name()='lod2Solid' or local-name()='lod2MultiSurface']//*[local-name()='posList']/text()")
            if not pos_lists:
                pos_lists = elem.xpath(".//*[local-name()='lod1Solid' or local-name()='lod1MultiSurface']//*[local-name()='posList']/text()")
            kept_any = False
            for pl in pos_lists:
                nums = np.fromstring(pl.strip(), sep=" ", dtype=np.float64)
                if nums.size < 9 or nums.size % 3 != 0:
                    continue
                pts = nums.reshape(-1, 3)  # PLATEAU JGD2011: [lat, lon, elev_m]
                if np.allclose(pts[0], pts[-1]):
                    pts = pts[:-1]
                if len(pts) < 3:
                    continue
                xs, zs = latlon_to_game_xz(pts[:, 0], pts[:, 1], cfg["origin_lat"], cfg["origin_lon"])
                cx, cz = float(xs.mean()), float(zs.mean())
                if not (x_min <= cx <= x_max and z_min <= cz <= z_max):
                    continue
                ys = pts[:, 2] * cfg["vertical_exag"]
                poly3d = np.column_stack([xs, ys, zs]).astype(np.float32)
                # 簡易ファン三角形分割（凸・準凸面向け高速処理）
                n = len(poly3d)
                all_verts.append(poly3d)
                for k in range(1, n - 1):
                    all_faces.append([v_offset, v_offset + k, v_offset + k + 1])
                v_offset += n
                kept_any = True
            if kept_any:
                bldg_count += 1
            elem.clear()
        stream.close()

    if not all_verts:
        print("  ⚠️ 指定PLATEAUファイル内に対象範囲の建物が見つかりませんでした")
        return None

    V = np.vstack(all_verts)
    F = np.asarray(all_faces, dtype=np.int32)
    mat = PBRMaterial(name="plateau_bldg", baseColorFactor=[236, 238, 240, 255], roughnessFactor=0.82, metallicFactor=0.04)
    m = trimesh.Trimesh(vertices=V, faces=F, visual=TextureVisuals(material=mat), process=False)
    m.fix_normals()
    print(f"  [PLATEAU] {bldg_count:,} 棟, 三角形数={len(F):,} を生成しました")
    return {"plateau": m}

print("✅ 建物生成ビルダーを定義しました")

In [ ]:
# @title 6. 江の島・八景島 `.glb` を生成して書き出し
GENERATED = {}

for vid in TARGET_VENUES:
    cfg = VENUES[vid]
    print("=" * 64)
    print(f"🏝️  生成開始: {cfg['label']} ({vid})")
    t0 = time.time()

    terrain_mesh, meta = build_terrain_mesh(cfg, TERRAIN_GRID_N, ORTHO_TEX_SIZE)
    scene = trimesh.Scene()
    scene.add_geometry(terrain_mesh, node_name=f"{vid}_terrain")

    bldg_meshes = None
    p_path = PLATEAU_PATHS.get(vid, "").strip()
    if p_path:
        bldg_meshes = load_plateau_citygml_meshes(p_path, cfg, meta)
    if bldg_meshes is None and INCLUDE_OSM_BUILDINGS:
        bldg_meshes = build_osm_meshes(cfg, meta, cache_key=vid)

    if bldg_meshes:
        for cat, bm in bldg_meshes.items():
            scene.add_geometry(bm, node_name=f"{vid}_bldg_{cat}")

    out_glb = f"/content/output/{vid}.glb"
    glb_bytes = scene.export(file_type="glb")
    with open(out_glb, "wb") as f:
        f.write(glb_bytes)

    size_mb = len(glb_bytes) / (1024 * 1024)
    total_tris = sum(len(g.faces) for g in scene.geometry.values())
    print(f"✅ 完了: {out_glb} | サイズ={size_mb:.2f} MB | 総三角形数={total_tris:,} | 所要={time.time()-t0:.1f}s")
    GENERATED[vid] = {"glb_path": out_glb, "size_mb": size_mb, "tris": total_tris, "meta": meta, "cfg": cfg}

In [ ]:
# @title 7. 見た目確認（A: 航空写真＋標高ヒートマップ確認）
for vid, info in GENERATED.items():
    meta = info["meta"]
    cfg = info["cfg"]
    fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))
    axes[0].imshow(meta["ortho_tex"])
    axes[0].set_title(f"{cfg['label']} - 地理院 航空写真テクスチャ")
    axes[0].axis("off")

    H = meta["H_grid"]
    im = axes[1].imshow(np.where(H > 0, H, np.nan), cmap="terrain")
    axes[1].set_title(f"{cfg['label']} - DEM5A 標高 [m] (総三角形 {info['tris']:,} / {info['size_mb']:.2f}MB)")
    axes[1].axis("off")
    plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()

In [ ]:
# @title 8. 見た目確認（B: Colab内 インタラクティブ Three.js 3Dプレビュー）
# レース海面 (0,0) からの実際の見え方や、上空俯瞰・近接アングルをマウス操作で確認できます
def show_threejs_glb_preview(generated_map):
    if not generated_map:
        print("プレビュー対象のモデルがありません")
        return
    payload = {}
    for vid, info in generated_map.items():
        cfg = info["cfg"]
        with open(info["glb_path"], "rb") as f:
            b64 = base64.b64encode(f.read()).decode("ascii")
        tgt_isl = cfg["preview_target"]
        payload[vid] = {
            "label": cfg["label"],
            "size_mb": round(info["size_mb"], 2),
            "tris": info["tris"],
            "b64": b64,
            "cam_race": cfg["preview_cam"],
            "cam_close": [tgt_isl[0] - 260.0, 95.0, tgt_isl[2] + 520.0],
            "cam_top": [tgt_isl[0], 680.0, tgt_isl[2] + 920.0],
            "target": tgt_isl,
        }

    first_vid = next(iter(payload.keys()))
    venue_btns = "".join(
        f'<button class="vbtn" data-vid="{vid}" style="padding:4px 10px;border:1px solid #475569;border-radius:4px;background:{"#0ea5e9" if vid == first_vid else "#1e293b"};color:#fff;font-weight:700;cursor:pointer;">{d["label"].split("（")[0]}</button>'
        for vid, d in payload.items()
    )

    html = f"""
    <div style="font-family:sans-serif;border:1px solid #1e3a5f;border-radius:8px;overflow:hidden;background:#0a1929;">
      <div style="display:flex;flex-wrap:wrap;align-items:center;justify-content:space-between;gap:8px;padding:8px 12px;background:#0f2744;color:#f3f6f9;font-size:13px;">
        <div style="display:flex;align-items:center;gap:8px;">
          {venue_btns}
          <span id="metaInfo" style="color:#cbd5e1;font-size:12px;"></span>
        </div>
        <div style="display:flex;gap:6px;">
          <button id="btnRace" style="padding:4px 10px;border:none;border-radius:4px;background:#c9382b;color:#fff;font-weight:700;cursor:pointer;">レース海面(0,0)視点</button>
          <button id="btnClose" style="padding:4px 10px;border:none;border-radius:4px;background:#2563eb;color:#fff;font-weight:700;cursor:pointer;">島クロースアップ</button>
          <button id="btnTop" style="padding:4px 10px;border:none;border-radius:4px;background:#334155;color:#fff;font-weight:700;cursor:pointer;">上空俯瞰</button>
        </div>
      </div>
      <div id="threeWrap" style="width:100%;height:540px;position:relative;"></div>
    </div>
    <script type="importmap">
    {{
      "imports": {{
        "three": "https://unpkg.com/three@0.160.0/build/three.module.js",
        "three/addons/": "https://unpkg.com/three@0.160.0/examples/jsm/"
      }}
    }}
    </script>
    <script type="module">
      import * as THREE from 'three';
      import {{ OrbitControls }} from 'three/addons/controls/OrbitControls.js';
      import {{ GLTFLoader }} from 'three/addons/loaders/GLTFLoader.js';

      const DATA = {json.dumps(payload)};
      let currentVid = {json.dumps(first_vid)};

      const wrap = document.getElementById('threeWrap');
      const metaEl = document.getElementById('metaInfo');
      const renderer = new THREE.WebGLRenderer({{ antialias: true }});
      renderer.setPixelRatio(Math.min(window.devicePixelRatio || 1, 2));
      renderer.setSize(wrap.clientWidth, wrap.clientHeight);
      renderer.outputColorSpace = THREE.SRGBColorSpace;
      renderer.toneMapping = THREE.ACESFilmicToneMapping;
      renderer.toneMappingExposure = 1.06;
      wrap.appendChild(renderer.domElement);

      const scene = new THREE.Scene();
      scene.background = new THREE.Color(0x88b8de);
      scene.fog = new THREE.FogExp2(0x9bc2de, 0.00018);

      const camera = new THREE.PerspectiveCamera(58, wrap.clientWidth / wrap.clientHeight, 1, 14000);
      const controls = new OrbitControls(camera, renderer.domElement);
      controls.enableDamping = true;
      controls.maxPolarAngle = Math.PI * 0.495;

      scene.add(new THREE.HemisphereLight(0xcbe6ff, 0x2a4a5e, 0.95));
      const sun = new THREE.DirectionalLight(0xfff4df, 1.85);
      sun.position.set(1200, 1400, 900);
      scene.add(sun);

      const sea = new THREE.Mesh(
        new THREE.PlaneGeometry(18000, 18000),
        new THREE.MeshStandardMaterial({{ color: 0x0d3b66, roughness: 0.28, metalness: 0.08 }})
      );
      sea.rotation.x = -Math.PI / 2;
      scene.add(sea);

      // レース海面原点(0, 0)のオレンジブイ目印
      const originMark = new THREE.Mesh(
        new THREE.ConeGeometry(8, 22, 12),
        new THREE.MeshStandardMaterial({{ color: 0xff5a1f, roughness: 0.35 }})
      );
      originMark.position.set(0, 11, 0);
      scene.add(originMark);

      const loader = new GLTFLoader();
      const loadedGroups = {{}};

      const setView = (pos, tgt) => {{
        camera.position.set(pos[0], pos[1], pos[2]);
        controls.target.set(tgt[0], tgt[1], tgt[2]);
        controls.update();
      }};

      const selectVenue = (vid) => {{
        currentVid = vid;
        const d = DATA[vid];
        metaEl.textContent = `${{d.label}} | ${{d.size_mb}} MB | ${{d.tris.toLocaleString()}} tris`;
        document.querySelectorAll('.vbtn').forEach((b) => {{
          b.style.background = b.dataset.vid === vid ? '#0ea5e9' : '#1e293b';
        }});
        Object.entries(loadedGroups).forEach(([k, g]) => {{ g.visible = (k === vid); }});
        if (!loadedGroups[vid]) {{
          const binStr = atob(d.b64);
          const bytes = new Uint8Array(binStr.length);
          for (let i = 0; i < binStr.length; i++) bytes[i] = binStr.charCodeAt(i);
          loader.parse(bytes.buffer, '', (gltf) => {{
            loadedGroups[vid] = gltf.scene;
            gltf.scene.visible = (currentVid === vid);
            scene.add(gltf.scene);
          }}, (err) => console.error(err));
        }}
        setView(d.cam_close, d.target);
      }};

      document.querySelectorAll('.vbtn').forEach((b) => {{
        b.onclick = () => selectVenue(b.dataset.vid);
      }});
      document.getElementById('btnRace').onclick = () => setView(DATA[currentVid].cam_race, DATA[currentVid].target);
      document.getElementById('btnClose').onclick = () => setView(DATA[currentVid].cam_close, DATA[currentVid].target);
      document.getElementById('btnTop').onclick = () => setView(DATA[currentVid].cam_top, DATA[currentVid].target);

      selectVenue(currentVid);
      const anim = () => {{
        requestAnimationFrame(anim);
        controls.update();
        renderer.render(scene, camera);
      }};
      anim();
    </script>
    """
    display(HTML(html))


show_threejs_glb_preview(GENERATED)

In [ ]:
# @title 9. 生成した `.glb` をPCへダウンロード（またはGoogle Driveへ保存）
# 気に入ったらこのセルを実行して enoshima.glb / hakkeijima.glb をダウンロードできます
SAVE_TO_DRIVE = False  # @param {type:"boolean"}
DOWNLOAD_NOW = False   # @param {type:"boolean"}

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    dst_dir = "/content/drive/MyDrive/sailing_venues_glb"
    os.makedirs(dst_dir, exist_ok=True)
    import shutil
    for vid, info in GENERATED.items():
        dst = os.path.join(dst_dir, f"{vid}.glb")
        shutil.copy2(info["glb_path"], dst)
        print("Driveへ保存:", dst)

if DOWNLOAD_NOW:
    from google.colab import files
    for vid, info in GENERATED.items():
        files.download(info["glb_path"])

print("出力ファイル一覧:")
for vid, info in GENERATED.items():
    print(f" - {info['glb_path']} ({info['size_mb']:.2f} MB, {info['tris']:,} tris)")